In [0]:
import requests

API_URL = "https://data.montgomerycountymd.gov/resource/xtyh-brr2.json"
BATCH_SIZE = 1000
MAX_BATCHES = 3

def retrieve(url, batch_size, max_batches):
    retrieved_records = []

    for i in range(max_batches):
        response = requests.get(url, timeout=30, params={"$limit": batch_size,"$offset": i * batch_size, "$order": "created DESC"})
        response.raise_for_status()

        retrieved_records.extend(response.json())

    return retrieved_records

records = retrieve(API_URL, BATCH_SIZE, MAX_BATCHES)
#print(records)


In [0]:
#turn ingested data into dataframe
df = spark.createDataFrame(records)
display(df)

In [0]:
df.printSchema()

In [0]:
#track when record was ingested convert to eastern time as new column
import pyspark.sql.functions as sf

df_bronze = df.withColumn(
    "ingestion_timestamp_eastern",
    sf.from_utc_timestamp(sf.current_timestamp(), "America/New_York").cast("timestamp")
)

display(df_bronze)

In [0]:
#save dataframe to bronze schema
#display(spark.sql("SHOW CATALOGS")) #get catalog name
#display(spark.sql("SHOW SCHEMAS")) #get schema name

bronze_table = "workspace.mc311_bronze.service_requests_raw"
df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)

In [0]:
#validate bronze schema
bronze_df = spark.table(bronze_table)
bronze_df.printSchema()